# Card106：固定局部时间标签通道（用户运行）

已采用单一方案：原生44cell的6bit时间ID编码成RM32chip＋16检测pilot；48个不交64D块QIM，δ=1/√12/α1，半径3/Hp≤1。channels4..15只写同步metadata，保留原多步4×8净32归因载荷，不终态重写payload。ECC/唯一解/pilot都不是身份认证。
四个VIEWED宿主2cal2eval；每臂OFF/P0/PT双钥，5个clean45窗及真实pixel删帧后center跨缝窗。完整4phase与10row都留；mixed/guard/crossseam合法ID不能当正确FULL单元。未知gain/offset不拟合救读，不按标签/消息择窗口/路径。
先保存cal全部P/64码距离；2400null P不按Hp/RM/真phase筛选，max阈值冻结后才处理eval。源信息只用于私有媒体形成/后评，blind只拿normalized latent的48投影/key/τ。先保存读回600tag表与288payload表，再开几何/消息真值。
实际最小完整预算：VAE1/D4；逻辑E288=旧80＋新208，encoder新2288；FFT104/vote208/新Counter16640，旧6400译值复用；PT full/cross encode/probe/decode各8。新source/DiT/gradient0，不读旧Q作新tag，不把复用当收益。5760局部＋1056原生row固定，native诊断不是媒体成功。
新P0→PT质量20对（3140帧/3120邻对），原OFF→P0质量20对复用。eval原载荷3840计划pair bits中944FULL，其他2896保留未定义cell目标（cross496）；不以tag筛payload或预测填缺失。
旧136/65负例、Card102有限正例及Card104 s02有限coarse/s04误检宽C/19vs40损失和35.51/34.74dB等质量成本保留。真实局部结果尚未知；合成7项core＋7项新接线检查只属工程。无状态空间、fresh来源或VAE训练。

旧cache不可读直接保留失败，无自动E/FFT重算、重试或强度/阈值scan。一次attempt先持久化，后续Run all只读已存JSON，不重新下载/安装/模型或续算。无B64/hash/manifest/版本/Gitclean门禁，无GPU型号硬门槛。GPU/模型/媒体真实调用由用户执行。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, subprocess
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Time-Label-Local-Channel-V1/fixed-viewed-stage1-v1')
def execution_artifacts(output):
    output=Path(output);found=[]
    for name in ('attempt.json','ledger.json','progress.json','result.json','thresholds.json','protocol.json','roles.json','plan.json','evaluation.json','post_truth_access.json'):
        for suffix in ('','.tmp'):
            if (output/(name+suffix)).exists() or (output/(name+suffix)).is_symlink():found.append(name+suffix)
    for name in ('private','native','raw','blind','payload','media','cached','receipts','markers','quality'):
        p=output/name
        if p.exists():found.extend(str(x.relative_to(output)) for x in p.rglob('*') if x.is_file() or x.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    SOURCE_ROOT=Path('/content/SC-SSTW-b-time-label-local-channel-v1')
    if not SOURCE_ROOT.exists():
        subprocess.run(['git','clone','--depth','1','--single-branch','--branch','dev/local-joint-state-payload-v1',
            'https://github.com/RICHAAARC/SC-SSTW.git',str(SOURCE_ROOT)],check=True)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/b_time_label_local_channel_v1_run.py').is_file():
        raise FileNotFoundError('Available source lacks fixed Stage1 runner: '+str(SOURCE_ROOT))
    OUTPUT.mkdir(parents=True,exist_ok=True);print('Available editable source',SOURCE_ROOT)
else:
    print('Existing attempt/artifacts: read saved results only; no source/setup/physical reentry')


固定执行，不选模式或试参数。仅修复实际缺失导入/媒体工具；失败或中断不重启worker。

In [ ]:

import subprocess, sys, time
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan VAE dependencies')
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','-qq','update'], 'Prepare media dependency')
        command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))
    with (OUTPUT/'runner.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'experiments.wan_state_clock.b_time_label_local_channel_v1_run', '--output', str(OUTPUT)],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    try:
        while process.poll() is None:
            time.sleep(30)
            try:
                progress = json.loads((OUTPUT/'progress.json').read_text())
                print('Units', progress['returned'], '/', progress['planned'], 'active', progress['active'], flush=True)
            except (OSError,ValueError,KeyError,TypeError) as error:
                print('Progress read unavailable; observing same worker:',type(error).__name__,str(error),flush=True)
    except KeyboardInterrupt:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
        print('Stopped; saved attempts consumed. Later Run all is read-only.')
        raise
    print('Runner return code',process.returncode,'; inspect saved results and runner.log. No retry.')
else:
    print('Existing attempt: source/setup/physical execution skipped.')


只读保存结果：检出、正确标签、错误合法标签、拒读、载荷退步和质量分别报告。

In [ ]:

for name in ('result.json','thresholds.json','evaluation.json'):
    p=OUTPUT/name
    if not p.is_file():print('Not available:',name);continue
    try:v=json.loads(p.read_text())
    except (OSError,ValueError) as e:print('Read failure',name,str(e));continue
    if name=='result.json':print('Execution',v.get('status'),'units',v.get('counts'),'fixed tag rows',v.get('row_status_counts'),'technical rows',v.get('technical_tag_rows'))
    elif name=='thresholds.json':
        print('Complete unfiltered pilot-null calibration',v.get('status'),'tau',v.get('tau'),'ID screening',v.get('ID_screening_used'))
        for host,stats in v.get('hosts',{}).items():
            maxima=stats.get('all_maxima',[])
            coords=[{k:row.get(k) for k in ('class','view','phase','row')} for row in maxima[:3]]
            print('Calibration host',host,'N',stats.get('N'),'null_rows',stats.get('null_rows'),'tied_maxima',len(maxima),'first3_coordinates',coords)
    else:
        print('Fixed denominator',v.get('fixed_denominators'))
        for scope,row in v.get('summary',{}).items():print('Tag scope',scope,row)
        for scope,row in v.get('strata',{}).items():
            if scope.startswith('evaluation/local/'):print('Local eval stratum',scope,row)
        print('Original payload side effect, no tag-selected path',v.get('payload_summary'))
for p in sorted((OUTPUT/'quality').rglob('*.json')) if (OUTPUT/'quality').is_dir() else ():
    if 'OFF_to_P0_reference' in str(p):continue
    try:
        v=json.loads(p.read_text());print('Quality',str(p.relative_to(OUTPUT)),v.get('whole'),v.get('psnr_db_range1'),'adjacent',v.get('adjacent_residual'),'PASS',v.get('quality_pass'))
    except (OSError,ValueError) as e:print('Read failure',str(p),str(e))
ledger={}
try:
    if (OUTPUT/'ledger.json').is_file():ledger=json.loads((OUTPUT/'ledger.json').read_text())
except (OSError,ValueError) as e:print('Ledger read failure',str(e))
for p in (OUTPUT/'receipts').rglob('*.json') if (OUTPUT/'receipts').is_dir() else ():
    try:
        v=json.loads(p.read_text());ledger[v['slot']]=v
    except (OSError,ValueError,KeyError,TypeError) as e:print('Receipt read failure',str(p),str(e))
bad=[(k,v.get('status'),v.get('error'),v.get('unavailable_from')) for k,v in ledger.items() if v.get('status')!='RETURNED']
print('Unreturned fixed units',len(bad),'/',len(ledger),'first20',bad[:20])
print('VIEWED local development only. Native exact code is not media evidence. Mixed/guard/crossseam legal labels are not correct temporal support.')
print('Logical E288 = old cached80 + newly run208. Old6400 decoded bits are reuse, not new observations or recovered capacity.')
print('No identity authentication/state-space/independent confirmation/quality PASS. All reject is not success. Full failures stay at',OUTPUT)
